# Synthetic Data Generation for Crochet Chart YOLO-OBB Detection (v3)

**Purpose**: Generate realistic synthetic crochet chart images with OBB labels for training.

**Key improvements in v3:**
- Procedural symbol drawing (supplements limited PNG templates)
- V-pattern grid generator (paired rotated doubles forming V-stitches)
- Triangular/expanding chart layouts (rows widen going up, matching real charts)
- Enhanced noise: circled row numbers, direction arrows, text annotations
- Color tinting to match real-data palettes (green, purple, pink)
- Vertical ensemble chain segments
- Background texture variation (grid lines, aging effects)
- Auto `data.yaml` generation

**9-class system:** chain(0), double(1), double treble(2), enseble_chain(3), fan(4), half_double(5), noise(6), single(7), treble(8)


In [18]:
import cv2 as cv
import numpy as np
import math
import os
import random
import yaml
from glob import glob
from pathlib import Path

# ═══════════════════════════════════════════════════════════════════════════════
# Configuration
# ═══════════════════════════════════════════════════════════════════════════════

# ──── Paths (adjust for Colab) ────
# On Colab, set PROJECT_DIR = "/content/drive/MyDrive/Crochet_data/project_yolo_obb"
PROJECT_DIR = os.path.abspath(".")
TEMPLATE_DIR = os.path.join(PROJECT_DIR, "data", "raw", "templates")
OUTPUT_DIR   = os.path.join(PROJECT_DIR, "training_data")

TRAIN_IMG = os.path.join(OUTPUT_DIR, "train", "images")
TRAIN_LBL = os.path.join(OUTPUT_DIR, "train", "labels")
VAL_IMG   = os.path.join(OUTPUT_DIR, "val", "images")
VAL_LBL   = os.path.join(OUTPUT_DIR, "val", "labels")

for d in [TRAIN_IMG, TRAIN_LBL, VAL_IMG, VAL_LBL]:
    os.makedirs(d, exist_ok=True)

CLASS_MAP = {
    "chain":          0,
    "double":         1,
    "double treble":  2,
    "enseble_chain":  3,
    "fan":            4,
    "half_double":    5,
    "noise":          6,
    "single":         7,
    "treble":         8,
}
NUM_CLASSES = len(CLASS_MAP)

# ──── Generation parameters ────
NUM_TRAIN = 300
NUM_VAL   = 60
IMG_SIZE  = 640  # output image size

# ──── Color palettes observed in real charts ────
PALETTES = {
    "green":  {"fg": (40, 120, 60),  "bg": (245, 248, 240)},
    "purple": {"fg": (90, 50, 130),  "bg": (248, 244, 252)},
    "pink":   {"fg": (160, 50, 80),  "bg": (252, 245, 248)},
    "black":  {"fg": (30, 30, 30),   "bg": (255, 255, 255)},
    "blue":   {"fg": (50, 70, 150),  "bg": (245, 248, 255)},
}

print(f"Project dir: {PROJECT_DIR}")
print(f"Templates:   {TEMPLATE_DIR}")
print(f"Output:      {OUTPUT_DIR}")
print(f"Classes:     {CLASS_MAP}")


Project dir: /Users/elevchenko/Documents/DataScience/Crochet
Templates:   /Users/elevchenko/Documents/DataScience/Crochet/data/raw/templates
Output:      /Users/elevchenko/Documents/DataScience/Crochet/training_data
Classes:     {'chain': 0, 'double': 1, 'double treble': 2, 'enseble_chain': 3, 'fan': 4, 'half_double': 5, 'noise': 6, 'single': 7, 'treble': 8}


In [19]:
# ═══════════════════════════════════════════════════════════════════════════════
# Template Loading
# ═══════════════════════════════════════════════════════════════════════════════

TEMPLATES = {}  # class_name -> list of BGRA images

for cls_name in CLASS_MAP:
    cls_dir = os.path.join(TEMPLATE_DIR, cls_name)
    TEMPLATES[cls_name] = []
    if os.path.isdir(cls_dir):
        for p in sorted(glob(os.path.join(cls_dir, "*.png"))):
            img = cv.imread(p, cv.IMREAD_UNCHANGED)
            if img is not None:
                if img.ndim == 2:
                    img = cv.cvtColor(img, cv.COLOR_GRAY2BGRA)
                elif img.shape[2] == 3:
                    img = cv.cvtColor(img, cv.COLOR_BGR2BGRA)
                TEMPLATES[cls_name].append(img)

# Also load t_o.png and t_x.png as potential noise/single templates
for extra in ["t_o.png", "t_x.png"]:
    p = os.path.join(TEMPLATE_DIR, extra)
    if os.path.isfile(p):
        img = cv.imread(p, cv.IMREAD_UNCHANGED)
        if img is not None:
            if img.ndim == 2:
                img = cv.cvtColor(img, cv.COLOR_GRAY2BGRA)
            elif img.shape[2] == 3:
                img = cv.cvtColor(img, cv.COLOR_BGR2BGRA)
            # t_x -> single cross variant, t_o -> noise circle variant
            if "x" in extra:
                TEMPLATES["single"].append(img)
            else:
                TEMPLATES["noise"].append(img)

for cls_name, imgs in TEMPLATES.items():
    print(f"  {cls_name}: {len(imgs)} templates")


  chain: 4 templates
  double: 10 templates
  double treble: 0 templates
  enseble_chain: 0 templates
  fan: 0 templates
  half_double: 1 templates
  noise: 1 templates
  single: 4 templates
  treble: 3 templates


In [20]:
# ═══════════════════════════════════════════════════════════════════════════════
# Procedural Symbol Drawing
# ═══════════════════════════════════════════════════════════════════════════════
# Supplements PNG templates — especially needed for classes with 0-1 templates
# (enseble_chain, fan, noise). Returns BGRA images.

def draw_chain(size=40, thickness=2, color=(0, 0, 0)):
    """Draw an oval/ellipse chain stitch symbol."""
    w, h = size, int(size * 0.55)
    img = np.zeros((h + 4, w + 4, 4), dtype=np.uint8)
    cx, cy = (w + 4) // 2, (h + 4) // 2
    cv.ellipse(img, (cx, cy), (w // 2, h // 2), 0, 0, 360, (*color, 255), thickness)
    return img

def draw_single(size=40, thickness=2, color=(0, 0, 0)):
    """Draw a + or x cross for single crochet."""
    s = size
    img = np.zeros((s, s, 4), dtype=np.uint8)
    m = s // 2
    # Vertical line with small horizontal bar
    cv.line(img, (m, 2), (m, s - 3), (*color, 255), thickness)
    cv.line(img, (m - s // 4, m), (m + s // 4, m), (*color, 255), thickness)
    return img

def draw_single_x(size=40, thickness=2, color=(0, 0, 0)):
    """Draw an X for single crochet variant."""
    s = size
    img = np.zeros((s, s, 4), dtype=np.uint8)
    pad = s // 5
    cv.line(img, (pad, pad), (s - pad, s - pad), (*color, 255), thickness)
    cv.line(img, (s - pad, pad), (pad, s - pad), (*color, 255), thickness)
    return img

def draw_double(size=80, thickness=2, color=(0, 0, 0)):
    """Draw a vertical line with a diagonal slash (double crochet)."""
    w = int(size * 0.35)
    h = size
    img = np.zeros((h, w, 4), dtype=np.uint8)
    mx = w // 2
    # Main vertical stroke
    cv.line(img, (mx, 2), (mx, h - 3), (*color, 255), thickness)
    # Diagonal slash across middle
    cv.line(img, (mx - w // 3, h // 2 + h // 6),
            (mx + w // 3, h // 2 - h // 6), (*color, 255), thickness)
    # Small T at top
    cv.line(img, (mx - w // 4, 2), (mx + w // 4, 2), (*color, 255), thickness)
    return img

def draw_half_double(size=60, thickness=2, color=(0, 0, 0)):
    """Draw vertical line with horizontal bar (half double crochet)."""
    w = int(size * 0.4)
    h = size
    img = np.zeros((h, w, 4), dtype=np.uint8)
    mx = w // 2
    cv.line(img, (mx, 2), (mx, h - 3), (*color, 255), thickness)
    # Horizontal bar at ~2/3 height
    y_bar = h // 3
    cv.line(img, (mx - w // 3, y_bar), (mx + w // 3, y_bar), (*color, 255), thickness)
    # T at top
    cv.line(img, (mx - w // 4, 2), (mx + w // 4, 2), (*color, 255), thickness)
    return img

def draw_treble(size=100, thickness=2, color=(0, 0, 0)):
    """Draw vertical line with two diagonal slashes (treble crochet)."""
    w = int(size * 0.35)
    h = size
    img = np.zeros((h, w, 4), dtype=np.uint8)
    mx = w // 2
    cv.line(img, (mx, 2), (mx, h - 3), (*color, 255), thickness)
    # Two slashes
    for offset in [-h // 8, h // 8]:
        cy = h // 2 + offset
        cv.line(img, (mx - w // 3, cy + h // 8),
                (mx + w // 3, cy - h // 8), (*color, 255), thickness)
    cv.line(img, (mx - w // 4, 2), (mx + w // 4, 2), (*color, 255), thickness)
    return img

def draw_double_treble(size=120, thickness=2, color=(0, 0, 0)):
    """Draw vertical line with three diagonal slashes (double treble crochet)."""
    w = int(size * 0.35)
    h = size
    img = np.zeros((h, w, 4), dtype=np.uint8)
    mx = w // 2
    cv.line(img, (mx, 2), (mx, h - 3), (*color, 255), thickness)
    # Three slashes spread across the stem
    for offset in [-h // 6, 0, h // 6]:
        cy = h // 2 + offset
        cv.line(img, (mx - w // 3, cy + h // 10),
                (mx + w // 3, cy - h // 10), (*color, 255), thickness)
    cv.line(img, (mx - w // 4, 2), (mx + w // 4, 2), (*color, 255), thickness)
    return img

def draw_fan(n_spokes=5, spoke_len=70, thickness=2, color=(0, 0, 0)):
    """Draw a fan/shell: multiple lines radiating from a single base point."""
    spread = math.radians(90)  # total angular spread
    w = int(spoke_len * 1.6)
    h = int(spoke_len * 1.2)
    img = np.zeros((h, w, 4), dtype=np.uint8)
    base = (w // 2, h - 4)
    for i in range(n_spokes):
        angle = math.pi / 2 + spread / 2 - (spread * i / max(n_spokes - 1, 1))
        ex = int(base[0] + spoke_len * math.cos(angle))
        ey = int(base[1] - spoke_len * math.sin(angle))
        cv.line(img, base, (ex, ey), (*color, 255), thickness)
        # Small T at tip
        perp_dx = int(5 * math.sin(angle))
        perp_dy = int(5 * math.cos(angle))
        cv.line(img, (ex - perp_dx, ey - perp_dy),
                (ex + perp_dx, ey + perp_dy), (*color, 255), thickness)
    return img

def draw_ensemble_chain(n_chains=5, chain_size=20, thickness=2, color=(0, 0, 0)):
    """Draw a vertical column of chain ovals (ensemble/foundation chain)."""
    ch_h = int(chain_size * 0.6)
    gap = 2
    total_h = n_chains * (ch_h + gap) + 4
    w = chain_size + 8
    img = np.zeros((total_h, w, 4), dtype=np.uint8)
    cx = w // 2
    for i in range(n_chains):
        cy = 2 + ch_h // 2 + i * (ch_h + gap)
        cv.ellipse(img, (cx, cy), (chain_size // 2, ch_h // 2), 0, 0, 360,
                   (*color, 255), thickness)
    return img

def draw_noise_circle(size=30, thickness=2, color=(0, 0, 0)):
    """Draw a small filled or hollow circle (noise marker)."""
    img = np.zeros((size, size, 4), dtype=np.uint8)
    r = size // 3
    cv.circle(img, (size // 2, size // 2), r, (*color, 255), thickness)
    return img

def draw_noise_number(num=1, size=30, color=(0, 0, 0)):
    """Draw a circled number (row counter noise)."""
    img = np.zeros((size, size, 4), dtype=np.uint8)
    r = size // 2 - 2
    cv.circle(img, (size // 2, size // 2), r, (*color, 255), 1)
    font = cv.FONT_HERSHEY_SIMPLEX
    txt = str(num)
    sc = size / 60
    (tw, th), _ = cv.getTextSize(txt, font, sc, 1)
    cv.putText(img, txt, (size // 2 - tw // 2, size // 2 + th // 2),
               font, sc, (*color, 255), 1, cv.LINE_AA)
    return img

def draw_noise_arrow(size=40, thickness=2, color=(0, 0, 0), direction="right"):
    """Draw a direction arrow (noise annotation)."""
    w, h = size, int(size * 0.4)
    img = np.zeros((h, w, 4), dtype=np.uint8)
    y = h // 2
    if direction == "right":
        cv.line(img, (2, y), (w - 6, y), (*color, 255), thickness)
        cv.line(img, (w - 10, y - 5), (w - 4, y), (*color, 255), thickness)
        cv.line(img, (w - 10, y + 5), (w - 4, y), (*color, 255), thickness)
    else:
        cv.line(img, (6, y), (w - 2, y), (*color, 255), thickness)
        cv.line(img, (10, y - 5), (4, y), (*color, 255), thickness)
        cv.line(img, (10, y + 5), (4, y), (*color, 255), thickness)
    return img

# Map class names to procedural draw functions
PROCEDURAL_DRAWERS = {
    "chain":          [draw_chain],
    "double":         [draw_double],
    "double treble":  [draw_double_treble],
    "single":         [draw_single, draw_single_x],
    "half_double":    [draw_half_double],
    "treble":         [draw_treble],
    "fan":            [draw_fan],
    "enseble_chain":  [draw_ensemble_chain],
    "noise":          [draw_noise_circle, draw_noise_number, draw_noise_arrow],
}

print("Procedural drawers registered for:", list(PROCEDURAL_DRAWERS.keys()))


Procedural drawers registered for: ['chain', 'double', 'double treble', 'single', 'half_double', 'treble', 'fan', 'enseble_chain', 'noise']


In [21]:
# ═══════════════════════════════════════════════════════════════════════════════
# Symbol Retrieval & Utilities
# ═══════════════════════════════════════════════════════════════════════════════

def get_symbol(cls_name, target_h, color=(0, 0, 0), thickness=2):
    """Get a symbol image (template or procedural), scaled to target height.
    Returns BGRA image."""
    use_template = random.random() < 0.5 and len(TEMPLATES.get(cls_name, [])) > 0

    if use_template:
        tmpl = random.choice(TEMPLATES[cls_name]).copy()
    else:
        drawers = PROCEDURAL_DRAWERS.get(cls_name, [])
        if not drawers:
            if TEMPLATES.get(cls_name):
                tmpl = random.choice(TEMPLATES[cls_name]).copy()
            else:
                # Fallback: small dot
                tmpl = np.zeros((20, 20, 4), dtype=np.uint8)
                cv.circle(tmpl, (10, 10), 5, (*color, 255), -1)
        else:
            drawer = random.choice(drawers)
            kwargs = {"color": color, "thickness": thickness}
            if cls_name == "chain":
                kwargs["size"] = random.randint(30, 50)
            elif cls_name == "double":
                kwargs["size"] = random.randint(60, 100)
            elif cls_name == "single":
                kwargs["size"] = random.randint(25, 45)
            elif cls_name == "half_double":
                kwargs["size"] = random.randint(45, 70)
            elif cls_name == "treble":
                kwargs["size"] = random.randint(80, 120)
            elif cls_name == "double treble":
                kwargs["size"] = random.randint(100, 140)
            elif cls_name == "fan":
                kwargs = {"n_spokes": random.randint(3, 7),
                          "spoke_len": random.randint(50, 80),
                          "thickness": thickness, "color": color}
            elif cls_name == "enseble_chain":
                kwargs = {"n_chains": random.randint(3, 8),
                          "chain_size": random.randint(15, 25),
                          "thickness": thickness, "color": color}
            elif cls_name == "noise":
                if drawer == draw_noise_number:
                    kwargs = {"num": random.randint(1, 30),
                              "size": random.randint(20, 35), "color": color}
                elif drawer == draw_noise_arrow:
                    kwargs = {"size": random.randint(30, 50), "thickness": thickness,
                              "color": color, "direction": random.choice(["left", "right"])}
                else:
                    kwargs = {"size": random.randint(15, 30),
                              "thickness": thickness, "color": color}
            tmpl = drawer(**kwargs)

    # Scale to target height
    if tmpl.shape[0] < 1 or tmpl.shape[1] < 1:
        tmpl = np.zeros((target_h, max(target_h // 2, 10), 4), dtype=np.uint8)
    else:
        scale = target_h / tmpl.shape[0]
        new_w = max(int(tmpl.shape[1] * scale), 1)
        tmpl = cv.resize(tmpl, (new_w, target_h), interpolation=cv.INTER_AREA)

    return tmpl


def tint_image(img, target_color):
    """Tint a BGRA image toward a target BGR color."""
    out = img.copy()
    alpha = out[:, :, 3]
    mask = alpha > 30
    for c in range(3):
        channel = out[:, :, c].astype(np.float32)
        channel[mask] = channel[mask] * 0.3 + target_color[c] * 0.7
        out[:, :, c] = np.clip(channel, 0, 255).astype(np.uint8)
    return out


def paste_symbol(canvas, symbol, cx, cy, angle_deg=0):
    """Paste a BGRA symbol onto a BGR canvas at (cx, cy) with rotation.
    Returns the 4-corner OBB coordinates (in pixel space)."""
    h, w = symbol.shape[:2]

    # Rotation matrix around symbol center
    M = cv.getRotationMatrix2D((w / 2, h / 2), -angle_deg, 1.0)
    cos_a = abs(M[0, 0])
    sin_a = abs(M[0, 1])
    new_w = int(h * sin_a + w * cos_a)
    new_h = int(h * cos_a + w * sin_a)
    M[0, 2] += (new_w - w) / 2
    M[1, 2] += (new_h - h) / 2

    rotated = cv.warpAffine(symbol, M, (new_w, new_h),
                            flags=cv.INTER_LINEAR, borderValue=(0, 0, 0, 0))

    # Compute paste region
    x1 = int(cx - new_w / 2)
    y1 = int(cy - new_h / 2)
    x2 = x1 + new_w
    y2 = y1 + new_h

    # Clip to canvas
    ch, cw = canvas.shape[:2]
    sx = max(0, -x1)
    sy = max(0, -y1)
    ex = min(new_w, cw - x1)
    ey = min(new_h, ch - y1)

    if sx >= ex or sy >= ey:
        return None  # fully outside

    roi = canvas[y1 + sy:y1 + ey, x1 + sx:x1 + ex]
    patch = rotated[sy:ey, sx:ex]
    alpha = patch[:, :, 3:4].astype(np.float32) / 255.0

    for c in range(3):
        roi[:, :, c] = (alpha[:, :, 0] * patch[:, :, c] +
                        (1 - alpha[:, :, 0]) * roi[:, :, c]).astype(np.uint8)

    # Compute OBB corners (original rect corners transformed)
    corners = np.array([
        [0, 0], [w, 0], [w, h], [0, h]
    ], dtype=np.float32)

    # Apply rotation
    ones = np.ones((4, 1), dtype=np.float32)
    pts = np.hstack([corners, ones])
    transformed = (M @ pts.T).T  # shape (4, 2)

    # Shift to canvas coords
    transformed[:, 0] += x1
    transformed[:, 1] += y1

    return transformed  # (4, 2) array of corner coords


def obb_to_yolo(corners, img_w, img_h):
    """Convert 4-corner pixel coords to YOLO OBB format (normalized)."""
    coords = []
    for pt in corners:
        x_norm = np.clip(pt[0] / img_w, 0, 1)
        y_norm = np.clip(pt[1] / img_h, 0, 1)
        coords.extend([float(x_norm), float(y_norm)])
    return coords


def add_grid_lines(canvas, spacing=30, color=(200, 200, 200), thickness=1):
    """Add faint grid lines to simulate chart paper."""
    h, w = canvas.shape[:2]
    for x in range(0, w, spacing):
        cv.line(canvas, (x, 0), (x, h), color, thickness)
    for y in range(0, h, spacing):
        cv.line(canvas, (0, y), (w, y), color, thickness)


def add_aging_texture(canvas, intensity=0.03):
    """Add subtle noise/aging to background."""
    noise = np.random.normal(0, intensity * 255, canvas.shape).astype(np.float32)
    result = np.clip(canvas.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    return result

print("Utilities loaded.")


Utilities loaded.


In [22]:
# ═══════════════════════════════════════════════════════════════════════════════
# Layout Generators
# ═══════════════════════════════════════════════════════════════════════════════
# Each generator returns (canvas_bgr, labels_list)
# where labels_list = [(class_id, x1,y1,x2,y2,x3,y3,x4,y4), ...] in normalized coords.

def generate_dense_rows(img_size=640, palette=None):
    """Dense horizontal rows of stitches — the most common chart layout."""
    if palette is None:
        palette = random.choice(list(PALETTES.values()))

    bg = np.full((img_size, img_size, 3), palette["bg"], dtype=np.uint8)
    fg = palette["fg"]

    if random.random() < 0.6:
        add_grid_lines(bg, spacing=random.randint(20, 40),
                       color=tuple(int(c * 0.9) for c in palette["bg"]))

    labels = []
    stitch_h = random.randint(25, 50)
    row_gap = stitch_h + random.randint(5, 15)
    margin = random.randint(20, 50)

    # Choose row stitches (weighted toward common classes)
    row_classes = random.choices(
        ["chain", "double", "single", "half_double", "treble"],
        weights=[5, 4, 3, 2, 1], k=random.randint(3, 8)
    )

    y = margin
    row_num = 0
    while y + stitch_h < img_size - margin:
        cls_name = row_classes[row_num % len(row_classes)]
        cls_id = CLASS_MAP[cls_name]

        # Slight row-level angle variation
        row_angle = random.gauss(0, 3)

        x = margin + random.randint(-5, 5)
        stitch_w = int(stitch_h * random.uniform(0.3, 0.8))
        gap_x = stitch_w + random.randint(2, 8)

        while x + stitch_w < img_size - margin:
            sym = get_symbol(cls_name, stitch_h, color=fg, thickness=random.randint(1, 2))
            angle = row_angle + random.gauss(0, 2)
            corners = paste_symbol(bg, sym, x + stitch_w // 2, y + stitch_h // 2, angle)
            if corners is not None:
                coords = obb_to_yolo(corners, img_size, img_size)
                labels.append((cls_id, *coords))
            x += gap_x + random.randint(-2, 2)

        # Occasionally add chain row between stitch rows
        if random.random() < 0.4:
            y += row_gap // 2
            cx = margin
            while cx + 20 < img_size - margin:
                ch_sym = get_symbol("chain", int(stitch_h * 0.5), color=fg)
                corners = paste_symbol(bg, ch_sym, cx + 10, y + stitch_h // 4, random.gauss(0, 5))
                if corners is not None:
                    coords = obb_to_yolo(corners, img_size, img_size)
                    labels.append((CLASS_MAP["chain"], *coords))
                cx += random.randint(12, 25)
            y += row_gap // 2

        y += row_gap
        row_num += 1

    # Add noise annotations on edges
    _add_edge_noise(bg, labels, img_size, fg, margin)

    bg = add_aging_texture(bg)
    return bg, labels


def _add_edge_noise(canvas, labels, img_size, fg, margin):
    """Add row numbers and arrows on chart edges."""
    for i in range(random.randint(0, 5)):
        ny = random.randint(margin, img_size - margin)
        side = random.choice(["left", "right"])
        nx = random.randint(2, margin - 5) if side == "left" else random.randint(img_size - margin + 5, img_size - 5)

        noise_type = random.choice(["number", "arrow", "circle"])
        if noise_type == "number":
            sym = draw_noise_number(num=i + 1, size=random.randint(18, 28), color=fg)
        elif noise_type == "arrow":
            sym = draw_noise_arrow(size=random.randint(25, 40), color=fg,
                                   direction="right" if side == "left" else "left")
        else:
            sym = draw_noise_circle(size=random.randint(12, 22), color=fg)

        corners = paste_symbol(canvas, sym, nx, ny, random.gauss(0, 3))
        if corners is not None:
            coords = obb_to_yolo(corners, img_size, img_size)
            labels.append((CLASS_MAP["noise"], *coords))


def generate_v_pattern_grid(img_size=640, palette=None):
    """V-stitch grid: pairs of rotated double crochets forming V shapes.
    This is the dominant pattern in the target image."""
    if palette is None:
        palette = random.choice(list(PALETTES.values()))

    bg = np.full((img_size, img_size, 3), palette["bg"], dtype=np.uint8)
    fg = palette["fg"]

    if random.random() < 0.5:
        add_grid_lines(bg, spacing=random.randint(25, 45),
                       color=tuple(int(c * 0.92) for c in palette["bg"]))

    labels = []
    stitch_h = random.randint(35, 55)
    v_spread = random.randint(15, 30)  # angle of V arms
    row_gap = stitch_h + random.randint(8, 20)
    col_gap = random.randint(25, 45)
    margin = random.randint(25, 50)

    y = margin + stitch_h // 2
    row_num = 0
    while y + stitch_h // 2 < img_size - margin:
        # Brick offset for alternating rows
        x_offset = (col_gap // 2) if row_num % 2 == 1 else 0
        x = margin + x_offset

        while x + col_gap < img_size - margin:
            # Left arm of V (rotated clockwise)
            angle_l = v_spread + random.gauss(0, 3)
            sym_l = get_symbol("double", stitch_h, color=fg, thickness=random.randint(1, 2))
            cx_l = x - col_gap // 6
            corners = paste_symbol(bg, sym_l, cx_l, y, angle_l)
            if corners is not None:
                labels.append((CLASS_MAP["double"], *obb_to_yolo(corners, img_size, img_size)))

            # Right arm of V (rotated counter-clockwise)
            angle_r = -v_spread + random.gauss(0, 3)
            sym_r = get_symbol("double", stitch_h, color=fg, thickness=random.randint(1, 2))
            cx_r = x + col_gap // 6
            corners = paste_symbol(bg, sym_r, cx_r, y, angle_r)
            if corners is not None:
                labels.append((CLASS_MAP["double"], *obb_to_yolo(corners, img_size, img_size)))

            # Chain at base of V
            if random.random() < 0.7:
                ch_sym = get_symbol("chain", int(stitch_h * 0.35), color=fg)
                corners = paste_symbol(bg, ch_sym, x, y + stitch_h // 3, random.gauss(0, 5))
                if corners is not None:
                    labels.append((CLASS_MAP["chain"], *obb_to_yolo(corners, img_size, img_size)))

            x += col_gap

        # Chain row between V-rows
        if random.random() < 0.5:
            chain_y = y + row_gap // 2
            cx = margin
            while cx + 15 < img_size - margin:
                ch = get_symbol("chain", int(stitch_h * 0.3), color=fg)
                corners = paste_symbol(bg, ch, cx + 8, chain_y, random.gauss(0, 8))
                if corners is not None:
                    labels.append((CLASS_MAP["chain"], *obb_to_yolo(corners, img_size, img_size)))
                cx += random.randint(10, 22)

        y += row_gap
        row_num += 1

    _add_edge_noise(bg, labels, img_size, fg, margin)
    bg = add_aging_texture(bg)
    return bg, labels


def generate_triangular_chart(img_size=640, palette=None):
    """Triangular/trapezoidal chart: rows expand upward (more stitches per row).
    Mimics the real target chart's expanding shape."""
    if palette is None:
        palette = random.choice(list(PALETTES.values()))

    bg = np.full((img_size, img_size, 3), palette["bg"], dtype=np.uint8)
    fg = palette["fg"]

    if random.random() < 0.5:
        add_grid_lines(bg, spacing=random.randint(20, 40),
                       color=tuple(int(c * 0.92) for c in palette["bg"]))

    labels = []
    stitch_h = random.randint(28, 45)
    row_gap = stitch_h + random.randint(5, 12)
    margin = random.randint(20, 40)

    # Build from bottom (narrow) to top (wide)
    n_rows = random.randint(6, 14)
    min_stitches = random.randint(2, 5)
    max_stitches = random.randint(15, 30)

    # Pattern type for rows
    pattern = random.choice(["v_stitch", "mixed", "single_type"])
    v_spread = random.randint(12, 25)

    for row_i in range(n_rows):
        # Number of stitches increases with row
        t = row_i / max(n_rows - 1, 1)
        n_stitches = int(min_stitches + t * (max_stitches - min_stitches))

        y = img_size - margin - row_i * row_gap
        if y - stitch_h // 2 < margin:
            break

        # Center the row horizontally
        row_width = n_stitches * (stitch_h * 0.6)
        x_start = (img_size - row_width) / 2
        stitch_gap = row_width / max(n_stitches, 1)

        for j in range(n_stitches):
            x = x_start + j * stitch_gap + stitch_gap / 2

            if pattern == "v_stitch" and random.random() < 0.7:
                # V-stitch pair
                ang_l = v_spread + random.gauss(0, 2)
                sym = get_symbol("double", stitch_h, color=fg)
                c = paste_symbol(bg, sym, int(x - 4), y, ang_l)
                if c is not None:
                    labels.append((CLASS_MAP["double"], *obb_to_yolo(c, img_size, img_size)))

                ang_r = -v_spread + random.gauss(0, 2)
                sym = get_symbol("double", stitch_h, color=fg)
                c = paste_symbol(bg, sym, int(x + 4), y, ang_r)
                if c is not None:
                    labels.append((CLASS_MAP["double"], *obb_to_yolo(c, img_size, img_size)))
            else:
                cls = random.choices(
                    ["double", "single", "chain", "half_double", "treble"],
                    weights=[4, 2, 3, 2, 1]
                )[0]
                sym = get_symbol(cls, stitch_h, color=fg)
                angle = random.gauss(0, 5)
                c = paste_symbol(bg, sym, int(x), y, angle)
                if c is not None:
                    labels.append((CLASS_MAP[cls], *obb_to_yolo(c, img_size, img_size)))

        # Foundation chain at bottom of each row section
        if row_i == 0 or random.random() < 0.3:
            chain_y = y + stitch_h // 2 + 5
            cx = x_start
            while cx < x_start + row_width:
                ch = get_symbol("chain", int(stitch_h * 0.35), color=fg)
                c = paste_symbol(bg, ch, int(cx), int(chain_y), random.gauss(0, 5))
                if c is not None:
                    labels.append((CLASS_MAP["chain"], *obb_to_yolo(c, img_size, img_size)))
                cx += random.randint(10, 20)

    # Ensemble chains on sides
    if random.random() < 0.4:
        for side_x in [margin // 2, img_size - margin // 2]:
            ec = get_symbol("enseble_chain", random.randint(60, 120), color=fg)
            c = paste_symbol(bg, ec, side_x, img_size // 2, random.gauss(0, 5))
            if c is not None:
                labels.append((CLASS_MAP["enseble_chain"], *obb_to_yolo(c, img_size, img_size)))

    _add_edge_noise(bg, labels, img_size, fg, margin)
    bg = add_aging_texture(bg)
    return bg, labels


def generate_fan_grid(img_size=640, palette=None):
    """Grid of fan/shell stitches with chain connections."""
    if palette is None:
        palette = random.choice(list(PALETTES.values()))

    bg = np.full((img_size, img_size, 3), palette["bg"], dtype=np.uint8)
    fg = palette["fg"]

    labels = []
    fan_h = random.randint(50, 80)
    row_gap = fan_h + random.randint(15, 30)
    col_gap = random.randint(60, 100)
    margin = random.randint(30, 50)

    y = margin + fan_h // 2
    row_num = 0
    while y + fan_h // 2 < img_size - margin:
        x_off = (col_gap // 2) if row_num % 2 == 1 else 0
        x = margin + x_off
        while x + col_gap // 2 < img_size - margin:
            n_spokes = random.randint(3, 7)
            sym = draw_fan(n_spokes=n_spokes, spoke_len=int(fan_h * 0.7),
                           thickness=random.randint(1, 2), color=fg)
            # Convert to BGRA if needed
            if sym.shape[2] == 4:
                pass
            angle = random.gauss(0, 5)
            corners = paste_symbol(bg, sym, x, y, angle)
            if corners is not None:
                labels.append((CLASS_MAP["fan"], *obb_to_yolo(corners, img_size, img_size)))

            # Chain connectors
            for ci in range(random.randint(1, 3)):
                ch_x = x + random.randint(-col_gap // 4, col_gap // 4)
                ch_y = y + fan_h // 2 + random.randint(5, 15)
                ch = get_symbol("chain", int(fan_h * 0.3), color=fg)
                c = paste_symbol(bg, ch, ch_x, ch_y, random.gauss(0, 10))
                if c is not None:
                    labels.append((CLASS_MAP["chain"], *obb_to_yolo(c, img_size, img_size)))

            x += col_gap
        y += row_gap
        row_num += 1

    _add_edge_noise(bg, labels, img_size, fg, margin)
    bg = add_aging_texture(bg)
    return bg, labels


def generate_mixed_chart(img_size=640, palette=None):
    """Mixed stitch chart with multiple stitch types in structured rows."""
    if palette is None:
        palette = random.choice(list(PALETTES.values()))

    bg = np.full((img_size, img_size, 3), palette["bg"], dtype=np.uint8)
    fg = palette["fg"]

    if random.random() < 0.4:
        add_grid_lines(bg, spacing=random.randint(20, 35),
                       color=tuple(int(c * 0.93) for c in palette["bg"]))

    labels = []
    stitch_h = random.randint(25, 45)
    row_gap = stitch_h + random.randint(8, 18)
    margin = random.randint(20, 45)

    y = margin
    while y + stitch_h < img_size - margin:
        # Pick 1-3 stitch types for this row
        row_types = random.choices(
            ["chain", "double", "single", "half_double", "treble", "fan"],
            weights=[4, 4, 3, 2, 1, 1],
            k=random.randint(1, 3)
        )

        x = margin + random.randint(-5, 10)
        while x + stitch_h < img_size - margin:
            cls = random.choice(row_types)
            sym = get_symbol(cls, stitch_h, color=fg)
            angle = random.gauss(0, 5)
            c = paste_symbol(bg, sym, x + stitch_h // 3, y + stitch_h // 2, angle)
            if c is not None:
                labels.append((CLASS_MAP[cls], *obb_to_yolo(c, img_size, img_size)))
            x += int(stitch_h * random.uniform(0.5, 1.0)) + random.randint(3, 10)

        y += row_gap

    # Add ensemble chains occasionally
    if random.random() < 0.3:
        ec = get_symbol("enseble_chain", random.randint(50, 100), color=fg)
        ex = random.choice([margin // 2, img_size - margin // 2])
        c = paste_symbol(bg, ec, ex, img_size // 2, random.gauss(0, 5))
        if c is not None:
            labels.append((CLASS_MAP["enseble_chain"], *obb_to_yolo(c, img_size, img_size)))

    _add_edge_noise(bg, labels, img_size, fg, margin)
    bg = add_aging_texture(bg)
    return bg, labels


def generate_chain_grid(img_size=640, palette=None):
    """Dense chain-only grid (foundation chain practice chart)."""
    if palette is None:
        palette = random.choice(list(PALETTES.values()))

    bg = np.full((img_size, img_size, 3), palette["bg"], dtype=np.uint8)
    fg = palette["fg"]

    labels = []
    ch_h = random.randint(15, 30)
    row_gap = ch_h + random.randint(5, 12)
    margin = random.randint(15, 35)

    y = margin
    while y + ch_h < img_size - margin:
        x = margin
        while x + ch_h < img_size - margin:
            sym = get_symbol("chain", ch_h, color=fg)
            angle = random.gauss(0, 8)
            c = paste_symbol(bg, sym, x + ch_h // 2, y + ch_h // 2, angle)
            if c is not None:
                labels.append((CLASS_MAP["chain"], *obb_to_yolo(c, img_size, img_size)))
            x += random.randint(ch_h, ch_h + 12)
        y += row_gap

    # Some noise
    for _ in range(random.randint(0, 4)):
        ny = random.randint(margin, img_size - margin)
        nx = random.choice([random.randint(2, margin), random.randint(img_size - margin, img_size - 5)])
        sym = draw_noise_number(num=random.randint(1, 20), size=20, color=fg)
        c = paste_symbol(bg, sym, nx, ny, 0)
        if c is not None:
            labels.append((CLASS_MAP["noise"], *obb_to_yolo(c, img_size, img_size)))

    bg = add_aging_texture(bg)
    return bg, labels


def generate_single_fan(img_size=640, palette=None):
    """Single large fan/shell with surrounding stitches."""
    if palette is None:
        palette = random.choice(list(PALETTES.values()))

    bg = np.full((img_size, img_size, 3), palette["bg"], dtype=np.uint8)
    fg = palette["fg"]

    labels = []

    # Large central fan
    fan_h = random.randint(120, 200)
    n_spokes = random.randint(5, 9)
    sym = draw_fan(n_spokes=n_spokes, spoke_len=int(fan_h * 0.7),
                   thickness=random.randint(2, 3), color=fg)
    cx, cy = img_size // 2, img_size // 2
    c = paste_symbol(bg, sym, cx, cy, random.gauss(0, 8))
    if c is not None:
        labels.append((CLASS_MAP["fan"], *obb_to_yolo(c, img_size, img_size)))

    # Surrounding stitches
    for _ in range(random.randint(10, 30)):
        cls = random.choices(
            ["chain", "double", "single", "half_double"],
            weights=[4, 3, 2, 2]
        )[0]
        sh = random.randint(20, 40)
        sx = random.randint(30, img_size - 30)
        sy = random.randint(30, img_size - 30)
        # Avoid center region
        if abs(sx - cx) < fan_h // 2 and abs(sy - cy) < fan_h // 2:
            continue
        sym = get_symbol(cls, sh, color=fg)
        c = paste_symbol(bg, sym, sx, sy, random.gauss(0, 15))
        if c is not None:
            labels.append((CLASS_MAP[cls], *obb_to_yolo(c, img_size, img_size)))

    bg = add_aging_texture(bg)
    return bg, labels


# Generator registry with weights (v-pattern and triangular weighted highest
# since they match the target chart)
GENERATORS = [
    (generate_v_pattern_grid,    0.30),  # V-stitch patterns (dominant in target)
    (generate_triangular_chart,  0.20),  # Expanding triangular layouts
    (generate_dense_rows,        0.20),  # Standard dense rows
    (generate_mixed_chart,       0.12),  # Mixed stitch types
    (generate_fan_grid,          0.08),  # Fan/shell grids
    (generate_chain_grid,        0.05),  # Chain-only grids
    (generate_single_fan,        0.05),  # Single large fan
]

gen_funcs, gen_weights = zip(*GENERATORS)
print(f"Registered {len(GENERATORS)} generators:")
for fn, w in GENERATORS:
    print(f"  {fn.__name__:30s} weight={w:.2f}")


Registered 7 generators:
  generate_v_pattern_grid        weight=0.30
  generate_triangular_chart      weight=0.20
  generate_dense_rows            weight=0.20
  generate_mixed_chart           weight=0.12
  generate_fan_grid              weight=0.08
  generate_chain_grid            weight=0.05
  generate_single_fan            weight=0.05


In [23]:
# ═══════════════════════════════════════════════════════════════════════════════
# Post-generation Augmentation
# ═══════════════════════════════════════════════════════════════════════════════

def augment_image(img):
    """Apply random augmentations to a generated chart image."""
    result = img.copy()

    # Random brightness/contrast
    if random.random() < 0.5:
        alpha = random.uniform(0.85, 1.15)  # contrast
        beta = random.randint(-15, 15)       # brightness
        result = np.clip(alpha * result.astype(np.float32) + beta, 0, 255).astype(np.uint8)

    # Random Gaussian blur
    if random.random() < 0.3:
        k = random.choice([3, 5])
        result = cv.GaussianBlur(result, (k, k), 0)

    # Random JPEG compression artifacts
    if random.random() < 0.3:
        quality = random.randint(50, 90)
        _, enc = cv.imencode('.jpg', result, [cv.IMWRITE_JPEG_QUALITY, quality])
        result = cv.imdecode(enc, cv.IMREAD_COLOR)

    # Random slight rotation (small, doesn't change labels much)
    if random.random() < 0.2:
        angle = random.uniform(-3, 3)
        h, w = result.shape[:2]
        M = cv.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        result = cv.warpAffine(result, M, (w, h), borderValue=(255, 255, 255))

    # Random scale (crop and resize)
    if random.random() < 0.2:
        scale = random.uniform(0.85, 1.0)
        h, w = result.shape[:2]
        nh, nw = int(h * scale), int(w * scale)
        y1 = random.randint(0, h - nh)
        x1 = random.randint(0, w - nw)
        crop = result[y1:y1+nh, x1:x1+nw]
        result = cv.resize(crop, (w, h))

    return result

print("Augmentation pipeline ready.")


Augmentation pipeline ready.


In [24]:
# ═══════════════════════════════════════════════════════════════════════════════
# Main Generation Loop
# ═══════════════════════════════════════════════════════════════════════════════

def generate_dataset(n_images, img_dir, lbl_dir, prefix="syn"):
    """Generate n_images synthetic charts with labels."""
    for i in range(n_images):
        # Pick generator
        gen_fn = random.choices(gen_funcs, weights=gen_weights, k=1)[0]
        palette = random.choice(list(PALETTES.values()))

        try:
            canvas, labels = gen_fn(img_size=IMG_SIZE, palette=palette)
        except Exception as e:
            print(f"  Warning: {gen_fn.__name__} failed: {e}")
            continue

        # Augment
        canvas = augment_image(canvas)

        # Filter invalid labels (all zeros, or degenerate boxes)
        valid_labels = []
        for lbl in labels:
            cls_id = lbl[0]
            coords = lbl[1:]
            # Check that the box has nonzero area
            xs = [coords[j] for j in range(0, 8, 2)]
            ys = [coords[j] for j in range(1, 8, 2)]
            w = max(xs) - min(xs)
            h = max(ys) - min(ys)
            if w > 0.005 and h > 0.005:  # at least ~3px at 640
                valid_labels.append(lbl)

        if len(valid_labels) == 0:
            continue

        # Save image
        fname = f"{prefix}_{i:04d}"
        cv.imwrite(os.path.join(img_dir, f"{fname}.png"), canvas)

        # Save labels (YOLO OBB format)
        with open(os.path.join(lbl_dir, f"{fname}.txt"), "w") as f:
            for lbl in valid_labels:
                cls_id = int(lbl[0])
                coords_str = " ".join(f"{c:.6f}" for c in lbl[1:])
                f.write(f"{cls_id} {coords_str}\n")

        if (i + 1) % 50 == 0:
            print(f"  Generated {i + 1}/{n_images} ({gen_fn.__name__})")

    print(f"Done: {n_images} images -> {img_dir}")


print(f"Generating {NUM_TRAIN} training images...")
generate_dataset(NUM_TRAIN, TRAIN_IMG, TRAIN_LBL, prefix="syn_train")

print(f"\nGenerating {NUM_VAL} validation images...")
generate_dataset(NUM_VAL, VAL_IMG, VAL_LBL, prefix="syn_val")


Generating 300 training images...
  Generated 50/300 (generate_dense_rows)
  Generated 100/300 (generate_v_pattern_grid)
  Generated 150/300 (generate_triangular_chart)
  Generated 200/300 (generate_v_pattern_grid)
  Generated 250/300 (generate_dense_rows)
  Generated 300/300 (generate_fan_grid)
Done: 300 images -> /Users/elevchenko/Documents/DataScience/Crochet/training_data/train/images

Generating 60 validation images...
  Generated 50/60 (generate_fan_grid)
Done: 60 images -> /Users/elevchenko/Documents/DataScience/Crochet/training_data/val/images


In [25]:
# ═══════════════════════════════════════════════════════════════════════════════
# Include Real Data in Training Set
# ═══════════════════════════════════════════════════════════════════════════════
# Copy real labeled data from project-11 into the training set for fine-tuning.
# The real data uses the same 9-class OBB format.

import shutil

REAL_DATA_DIR = os.path.join(PROJECT_DIR, "..", "project-11-at-2026-04-10-14-28-21a55b58")

# Check if real data exists
if os.path.isdir(REAL_DATA_DIR):
    real_images = sorted(glob(os.path.join(REAL_DATA_DIR, "images", "*.png")) +
                         glob(os.path.join(REAL_DATA_DIR, "images", "*.jpg")))
    real_labels = sorted(glob(os.path.join(REAL_DATA_DIR, "labels", "*.txt")))

    print(f"Found {len(real_images)} real images and {len(real_labels)} real label files")

    # Split: ~75% train, ~25% val
    n_val = max(1, len(real_images) // 4)
    indices = list(range(len(real_images)))
    random.shuffle(indices)
    val_idx = set(indices[:n_val])

    copied_train = 0
    copied_val = 0

    for idx, img_path in enumerate(real_images):
        stem = Path(img_path).stem
        lbl_path = os.path.join(REAL_DATA_DIR, "labels", stem + ".txt")

        if not os.path.isfile(lbl_path):
            continue

        if idx in val_idx:
            dst_img = os.path.join(VAL_IMG, f"real_{stem}.png")
            dst_lbl = os.path.join(VAL_LBL, f"real_{stem}.txt")
            copied_val += 1
        else:
            dst_img = os.path.join(TRAIN_IMG, f"real_{stem}.png")
            dst_lbl = os.path.join(TRAIN_LBL, f"real_{stem}.txt")
            copied_train += 1

        shutil.copy2(img_path, dst_img)
        shutil.copy2(lbl_path, dst_lbl)

    print(f"Copied {copied_train} real images to train, {copied_val} to val")
else:
    print(f"Real data dir not found: {REAL_DATA_DIR}")
    print("Skipping real data inclusion (synthetic only)")


Real data dir not found: /Users/elevchenko/Documents/DataScience/Crochet/../project-11-at-2026-04-10-14-28-21a55b58
Skipping real data inclusion (synthetic only)


In [26]:
# ═══════════════════════════════════════════════════════════════════════════════
# Generate data.yaml
# ═══════════════════════════════════════════════════════════════════════════════

data_yaml = {
    "path": OUTPUT_DIR,
    "train": "train/images",
    "val": "val/images",
    "names": {v: k for k, v in CLASS_MAP.items()}
}

yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print(f"Written: {yaml_path}")
print()
with open(yaml_path) as f:
    print(f.read())


Written: /Users/elevchenko/Documents/DataScience/Crochet/training_data/data.yaml

path: /Users/elevchenko/Documents/DataScience/Crochet/training_data
train: train/images
val: val/images
names:
  0: chain
  1: double
  2: double treble
  3: enseble_chain
  4: fan
  5: half_double
  6: noise
  7: single
  8: treble



In [27]:
# ═══════════════════════════════════════════════════════════════════════════════
# Verification & Preview
# ═══════════════════════════════════════════════════════════════════════════════

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Count files
train_imgs = glob(os.path.join(TRAIN_IMG, "*.png"))
val_imgs   = glob(os.path.join(VAL_IMG, "*.png"))
train_lbls = glob(os.path.join(TRAIN_LBL, "*.txt"))
val_lbls   = glob(os.path.join(VAL_LBL, "*.txt"))

print(f"Training:   {len(train_imgs)} images, {len(train_lbls)} labels")
print(f"Validation: {len(val_imgs)} images, {len(val_lbls)} labels")

# Class distribution
from collections import Counter
class_counts = Counter()
for lbl_path in train_lbls + val_lbls:
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 9:
                cls_id = int(parts[0])
                class_counts[cls_id] += 1

id_to_name = {v: k for k, v in CLASS_MAP.items()}
print("\nClass distribution:")
for cls_id in sorted(class_counts.keys()):
    print(f"  {cls_id} ({id_to_name.get(cls_id, '?'):15s}): {class_counts[cls_id]}")

# Preview grid
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
samples = random.sample(train_imgs, min(8, len(train_imgs)))
for ax, img_path in zip(axes.flat, samples):
    img = cv.imread(img_path)
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(Path(img_path).stem[:20], fontsize=8)
    ax.axis("off")
plt.suptitle("Synthetic Data Samples (v3)", fontsize=14)
plt.tight_layout()
preview_path = os.path.join(OUTPUT_DIR, "preview_grid.png")
plt.savefig(preview_path, dpi=100)
plt.close()
print(f"\nPreview saved: {preview_path}")


Training:   300 images, 300 labels
Validation: 60 images, 60 labels

Class distribution:
  0 (chain          ): 60069
  1 (double         ): 47144
  3 (enseble_chain  ): 62
  4 (fan            ): 2278
  5 (half_double    ): 4401
  6 (noise          ): 834
  7 (single         ): 5471
  8 (treble         ): 2125

Preview saved: /Users/elevchenko/Documents/DataScience/Crochet/training_data/preview_grid.png
